# Semantic Search & AI/RAG Integration

This notebook demonstrates advanced AI features of the `indianconstitution` library:

1. **Semantic Search** — sentence-transformers based contextual retrieval
2. **RAG Pipeline** — building constitutional context for LLM prompting
3. **Hybrid Search** — combining keyword + semantic for best results

**Requirements:**
```bash
pip install "indianconstitution[ai]"
```

In [ ]:
# Uncomment to install:
# !pip install -q "indianconstitution[ai]"

In [ ]:
from indianconstitution import get_constitution

ic = get_constitution()
print(f"Loaded: {ic}")

## 1. Semantic Search

Unlike keyword search, semantic search understands **meaning**.
It uses `all-MiniLM-L6-v2` embeddings to find conceptually relevant articles
even when the exact words don't appear.

In [ ]:
# Semantic search — finds conceptually related articles
results = ic.semantic_search(
    "protection against arbitrary state action",
    top_k=5
)

for r in results:
    print(f"[{r.number}] {r.title}  (score: {r.score:.4f})")

In [ ]:
# Compare: keyword search vs semantic search
query = "children's right to learn"

print(f"Query: '{query}'")
print("\n--- Keyword Search ---")
kw_results = ic.search(query, limit=3)
if kw_results:
    for r in kw_results:
        print(f"  [{r.number}] {r.title}")
else:
    print("  (no keyword matches)")

print("\n--- Semantic Search ---")
sem_results = ic.semantic_search(query, top_k=3)
for r in sem_results:
    print(f"  [{r.number}] {r.title}  (score: {r.score:.4f})")

## 2. RAG Pipeline Integration

Build grounded context blocks for Large Language Model prompting.

In [ ]:
def build_rag_context(query: str, top_k: int = 3) -> str:
    """Build a constitutional context block for LLM prompting."""
    results = ic.search(query, limit=top_k)
    context_blocks = []
    for article in results:
        context_blocks.append(
            f"**Article {article.number} \u2014 {article.title}**\n"
            f"{article.text}\n"
        )
    return "\n---\n".join(context_blocks)


context = build_rag_context("right to life and personal liberty")
print(context[:500])

In [ ]:
def build_semantic_rag_context(query: str, top_k: int = 3) -> str:
    """Build RAG context using semantic search for better recall."""
    results = ic.semantic_search(query, top_k=top_k)
    blocks = []
    for r in results:
        blocks.append(
            f"**Article {r.number} \u2014 {r.title}** (relevance: {r.score:.3f})\n"
            f"{r.text}\n"
        )
    return "\n---\n".join(blocks)


context = build_semantic_rag_context("can the government restrict speech?")
print(context[:500])

In [ ]:
# Full LLM prompt template
query = "What are the fundamental rights of Indian citizens?"
context = build_semantic_rag_context(query, top_k=5)

prompt = f"""You are a constitutional law expert. Answer the question
using ONLY the constitutional provisions below. Cite article numbers.

CONSTITUTIONAL CONTEXT:
{context}

QUESTION: {query}

ANSWER:"""

print(f"Prompt length: {len(prompt)} characters")
print("\n" + prompt[:800] + "...")

## 3. Hybrid Search Strategy

Combine keyword precision with semantic recall.

In [ ]:
def hybrid_search(query: str, top_k: int = 5):
    """Combine keyword + semantic search, deduplicate by article number."""
    # Keyword results (high precision)
    kw = {a.number: a for a in ic.search(query, limit=top_k)}
    # Semantic results (high recall)
    sem = {r.number: r for r in ic.semantic_search(query, top_k=top_k)}
    
    # Merge: keyword results first, then semantic-only
    merged = dict(kw)
    for num, art in sem.items():
        if num not in merged:
            merged[num] = art
    
    return list(merged.values())[:top_k]


query = "right to education"
results = hybrid_search(query, top_k=5)
print(f"Hybrid search for '{query}':")
for r in results:
    print(f"  [{r.number}] {r.title}")